# Token ratios, clipping and the exact KL gradient

Day 22 · Chapter 13 · CPU worked notebook

Prerequisites: softmax, Chapter12 PPO. Read Chapter13 sections13.3–13.5.


### Prediction

Predict: which side of the ratio interval stops further improvement for a negative advantage? Does clipping bound every next-token probability?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


### Prediction

Perturb epsilon or switch advantage sign before rerunning. Evidence boundary: exact categorical gradients at fixed states; no guarantee of small trajectory KL. Worked questions4–7.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
from dongxi_llms.grpo_lab import clipped_objective, exact_kl
ratios = torch.linspace(.3, 1.7, 101)
fig, ax = plt.subplots()
for advantage in (1., -1.):
    surrogate = torch.minimum(ratios*advantage, ratios.clamp(.8, 1.2)*advantage)
    ax.plot(ratios, surrogate, label=f"Advantage {advantage:+.0f}")
ax.axvline(.8, color="gray", linestyle=":")
ax.axvline(1.2, color="gray", linestyle=":")
ax.set(xlabel="Current / behavior chosen-token probability", ylabel="Clipped ascent surrogate")
ax.legend()
plt.show()

### Reference explanation

Reference explanation: positive advantages stop gaining at the upper boundary; negative ones stop gaining at the lower boundary. Movement in the unfavorable direction remains penalized. The plot describes a sampled objective, not a hard constraint on the whole distribution.

**Exercise modification reference.** In a working copy of original code3, set epsilon=.4; replace ratios.clamp(.8,1.2) with ratios.clamp(1-epsilon,1+epsilon), and change both vertical boundary lines to 1-epsilon and 1+epsilon. Keep ratios=torch.linspace(.3,1.7,101) and both advantage signs. Code5's ratio-one gradient probe stays unchanged and is insensitive to epsilon.

Upper positive-advantage plateau moves from1.2 to1.4; lower negative-advantage plateau moves from.8 to.6. Exact-KL interpretation belongs after code5. This concrete CPU variant has not been executed by the unchanged reference cell; it supplies an exercise reference rather than a new measured result.


Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Compare this fixed preview with your prediction and the current plot.

![Saved reference 1: token ratios kl](../figures/chapter-13/day-22-02_token_ratios_kl-01.png)

### Prediction

Exercise: derive the exact forward-KL gradient p_i(log(p_i/q_i)-KL). Test that the frozen reference receives no gradient.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
z = torch.tensor([.2, -.4, .8], dtype=torch.float64, requires_grad=True)
reference = torch.tensor([.7, -.1, .1], dtype=torch.float64, requires_grad=True)
kl = exact_kl(z, reference)
kl.backward()
p = z.detach().softmax(-1)
analytical = p*(z.detach().log_softmax(-1)-reference.detach().log_softmax(-1)-kl.detach())
torch.testing.assert_close(z.grad, analytical)
print("KL nats:", float(kl.detach()), "gradient:", z.grad.tolist())
print("Reference gradient:", reference.grad)
logp = torch.tensor([[.0], [.0]], requires_grad=True)
loss = clipped_objective(logp, torch.zeros_like(logp), torch.tensor([1., -1.]),
                         torch.ones_like(logp, dtype=torch.bool))
loss.backward()
print("Zero objective value / nonzero slope:", float(loss.detach()), logp.grad.flatten().tolist())

### Reference explanation

Interpretation: normalized group advantages can cancel in value while token-specific derivatives survive. The exact KL includes changing distribution weights. A sampled KL value identity does not automatically specify its full gradient.

The analytical exact forward-KL derivative includes the change of the policy weights as well as the log ratio. The reference is detached, and its gradient stays absent. This finite-state identity does not enforce a trajectory-wide KL limit.
